# LexChain parser benchmark — Docling vs MinerU vs Marker on OHR-Bench (Law)

**Kaggle setup:** GPU T4 x2 accelerator, internet ON.

Flow: **Cell 1** setup → **Cell 2** 3-document smoke test + full-run time estimate → *approve* → **Cell 3** full run (95 docs) → **Cell 4** metrics table + CSV.

**If the session dies:** progress is checkpointed after every document in `/kaggle/working/results`. In a new session, add the previous run's output as an *Input dataset*, then pass `--restore-from /kaggle/input/<slug>/results` in Cell 3.

In [ ]:
# Cell 1 — clone repo + set up envs, data, models (~15-25 min first time)
REPO_URL = "https://github.com/MarvinPescos/lexchain-parser-bench.git"

import os
if not os.path.exists("/kaggle/working/lexchain-parser-bench"):
    !git clone {REPO_URL} /kaggle/working/lexchain-parser-bench
%cd /kaggle/working/lexchain-parser-bench
!git pull
!nvidia-smi -L
!bash setup.sh

In [ ]:
# Cell 2 — smoke test: 3 smallest PDFs through all 3 tools, then estimate the full run
!python run_benchmark.py --limit 3
!python evaluate.py
!python estimate_runtime.py

**STOP — review the estimate above before running the full benchmark.**

Resuming a previous session? Attach its output as an input dataset and use:
`!python run_benchmark.py --restore-from /kaggle/input/<slug>/results`

In [ ]:
# Cell 3 — full run (resumable; smoke-test docs are skipped automatically)
!python run_benchmark.py

In [ ]:
# Cell 4 — evaluate + paper table (CSV + markdown saved to /kaggle/working/results)
!python evaluate.py

import pandas as pd
from IPython.display import display
display(pd.read_csv("/kaggle/working/results/results_summary.csv"))
print(open("/kaggle/working/results/results.md").read())

---
## Part 2 — Simulated-scanned condition

The Law set is 88 digital-born / 7 natively-scanned PDFs, but LexChain's real workload is mostly scanned. The cells below rasterize **all 95 PDFs at 200 DPI into image-only PDFs** (no text layer, identical filenames → ground truth maps 1:1) and rerun the benchmark with every tool forced through its **OCR path**. Tool configs are identical to Part 1 — the input condition is the only variable.

Results go to `/kaggle/working/results_scanned` (Part 1 results are untouched). Same checkpoint/resume behavior; to resume a dead session: `--restore-from /kaggle/input/<slug>/results_scanned`.

In [ ]:
# Cell 5 — build the simulated-scanned set (resumable; ~10-20 min, CPU only)
!python make_scanned.py

In [ ]:
# Cell 6 — scanned-condition smoke test (3 docs) + full-run estimate
# OCR mode is typically 3-6x slower than the digital condition — check the
# projection against Kaggle's ~12 h session cap before running Cell 7.
!python run_benchmark.py --condition scanned --limit 3
!python estimate_runtime.py --results-dir /kaggle/working/results_scanned --label scanned

**STOP — review the scanned-condition estimate above before the full OCR run.** If it exceeds the session cap, run Cell 7 in chunks across sessions (checkpointed, resumable via `--restore-from`).

In [ ]:
# Cell 7 — full scanned-condition run (resumable; smoke docs skipped)
!python run_benchmark.py --condition scanned

In [ ]:
# Cell 8 — combined evaluation: digital-born (88) vs natively-scanned (7) vs
# simulated-scanned (95), per tool, + ranking-stability check
!python evaluate.py --detect-scanned
!python compare_conditions.py

import pandas as pd
from IPython.display import display
display(pd.read_csv("/kaggle/working/results/comparison.csv"))
print(open("/kaggle/working/results/comparison.md").read())